In [498]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col,\
                            trim, upper, avg,\
                            sum as spark_sum, to_date, when, count,\
                            current_timestamp, regexp_replace, regexp,\
                            broadcast, round as spark_round, max as spark_max, mode,\
                            lit
from pyspark.sql.types import StructType, StructField, DoubleType, StringType, IntegerType, FloatType

In [385]:
# Create spark create function
@staticmethod
def create_spark_session():
    spark = SparkSession.builder\
                         .appName('Weather_Mini_Assignment_Practice_Session')\
                         .getOrCreate()
    return spark

In [392]:
class sparkFunctions:
    """
    This class 
        read CSV file, 
        standardize the columns,
        remove duplicates,
        remove unwanted columns,
        remove unwanted rows
        enrich data - adding additional columns
    """
    def __init__(self, spark):
        self.spark = spark
    def read_csv_file(self, input_file, schema=None):
        """
        This function will read the data from comma seperated file.
            1) It can process the file without schema
            2) It can process the file, if we pass the schema as arguments
        """
        if schema == None:
            df = self.spark.read.csv(input_file, inferSchema=True, header=True)
        else:
            df = self.spark.read.option("header", True).schema(schema).csv(input_file)
        return df
    def read_tsv_file(self, input_file, schema=None):
        """
        This function will read the data from tab seperated file.
            1) It can process the file without schema
            2) It can process the file, if we pass the schema as arguments
        """
        if schema == None:
            df = self.spark.read.csv(input_file, sep='\t', inferSchema=True, header=True)
        else:
            df = self.spark.read.option("header", True).schema(schema).csv(input_file)
        return df
    def remove_specific_strings_from_numeric(self, df, column_dict):
        """
        This function will update the invalid numbers as None for the numeric columns
            1) Arguments: df --> dataframe to be cleaned
                          col_list --> Numeric column list 
        """
        standardized_df = df
        for column_key,column_value in column_dict.items():
            standardized_df = standardized_df.withColumn(column_key,regexp_replace(col(column_key), column_value, ""))
        return standardized_df
    def remove_any_string_from_numeric(self, df, col_list):
        """
        This function will update the invalid numbers as None for the numeric columns
            1) Arguments: df --> dataframe to be cleaned
                          col_list --> Numeric column list 
        """
        standardized_df = df
        for column in col_list:
            standardized_df = standardized_df.withColumn(column,
                                             when(col(column).rlike("^[^0-9]+$"), None).otherwise(col(column))
                                           )
        return standardized_df
    def remove_spaces_from_string(self, df, col_list):
        """
        This function will remove the leading and trailing spaces from string columns
            1) Arguments: df --> dataframe to be cleaned
                          col_list --> Numeric column list 
        """
        standardized_df = df
        for column in col_list:
            standardized_df = standardized_df.withColumn(column,
                                             trim(col(column))
                                           )
        return standardized_df
    def standardize_null_values(self, df, column_dict):
        """
        This function fill NULL with dictionary values
            1) Arguments: df --> dataframe to be cleaned
                          column_dict --> column name and its fill value
        """
        standardized_df = df
        standardized_df = standardized_df.fillna(column_dict)
        return standardized_df
    def standardize_datatypes(self, df, column_dict):
        """
        This function fill NULL with dictionary values
            1) Arguments: df --> dataframe to be cleaned
                          column_dict --> column name and its fill value
        """
        standardized_df = df
        for column_name, column_type in column_dict.items():
            standardized_df = standardized_df.withColumn(column_name, col(column_name).cast(column_type))
        return standardized_df
        

In [417]:
class MiniAssignmentQuestions:
    """
    This class helps to retrive the answers to the mini assignment questions
    """
    def __init__(self, df):
        self.df = df
    def days_Rained_Next_Day(self):
    ###  Write a function that returns the number of days when it rained the next day. 
    ###  Hint: Count the number of rows where RainTomorrow is ‘Yes’.
        df_day_count = self.df.select(col('RainTomorrow'))\
                              .filter(col('RainTomorrow') == 'Yes')\
                              .groupBy(col('RainTomorrow'))\
                              .count().select('count')\
                              .toPandas()['count'][0]
        return df_day_count
    def avg_Sunshine_No_Rainfall(self):
    ###  Write a function that returns the average sunshine duration on days with no rainfall. 
    ###  Hint: Filter rows where Rainfall is 0 and then calculate the average of Sunshine.
        df_avg_sunshine = self.df.select(col('Sunshine'),col('Rainfall'))\
                                 .filter(col('Rainfall') == 0)\
                                 .agg(spark_round(avg("Sunshine"),2).alias("average"))\
                                 .select('average').toPandas()['average'][0]
        return df_avg_sunshine
    def max_temp_evening(self):
    ###  Write a function that returns the maximum temperature recorded at 3 PM.
    ###  Hint: Use the Temp3pm column and return the max value.  
        df_max_temp = self.df.select(col('Temp3pm'))\
                                 .agg(spark_round(spark_max("Temp3pm"),2).alias("maximum_temp"))\
                                 .select('maximum_temp').toPandas()['maximum_temp'][0]
        return df_max_temp
    def avg_evening_humidity_next_day_rain(self):
    ###  Write a function that returns the average humidity at 3 PM on days it rained the next day.
    ###  Hint: Filter rows where RainTomorrow is ‘Yes’ and then calculate the mean of Humidity3pm.
        df_avg_humid = self.df.select(col('Humidity3pm'))\
                                 .filter(col('RainTomorrow') == 'Yes')\
                                 .agg(spark_round(avg("Humidity3pm"),2).alias("average_humid"))\
                                 .select('average_humid').toPandas()['average_humid'][0]
        return df_avg_humid
    def cloudy_day_wind_direction(self):
    ###  Write a function that returns the most common wind direction at 9 AM on cloudy days. 
    ###  Hint: Filter rows where Cloud9am is above a threshold (e.g. >5) and then find the mode of WindDir9am. 
        df_mode_wind_dir = self.df.select(col('WindDir9am'))\
                                 .filter(col('Cloud9am') > 5)\
                                 .agg(mode("WindDir9am").alias("mode"))\
                                 .select('mode').toPandas()['mode'][0]
        return df_mode_wind_dir
    def total_revenue(self):
    ###  Write a function that returns the total revenue generated from all orders.
    ###  Hint: Multiply quantity by item_price for each row and then add the results.
        df_total_revenue = self.df.select(col('total_revenue'))\
                                 .agg(spark_sum("total_revenue").alias("total"))\
                                 .select('total').toPandas()['total'][0]
        return df_total_revenue

In [420]:
# Create spark session
SparkFunction = sparkFunctions(spark=create_spark_session())

In [421]:
# Read data files
input_csv_file='Data/Weather Dataset - CSV(in).csv'
input_tsv_file='Data/chipotle.tsv'
#schema = SparkFunction.define_schema()
#tsv_schema = SparkFunction.define_tsv_schema(
df_csv = SparkFunction.read_csv_file(input_csv_file)
df_tsv = SparkFunction.read_tsv_file(input_tsv_file)

In [422]:
####### Cleaning #######
# Remove specific alphabets in numeric columns
numeric_issue_column_dict = {'Sunshine':'NA', 'WindGustSpeed':'NA', 'WindSpeed9am':'NA'}
df_csv_cleaned = SparkFunction.remove_specific_strings_from_numeric(df_csv, numeric_issue_column_dict)
numeric_issue_column_dict = {'item_price':'\$'}
df_tsv_cleaned = SparkFunction.remove_specific_strings_from_numeric(df_tsv, numeric_issue_column_dict)
# Remove any alphabets in numeric columns

# Remove extra spaces in string columns
column_list=['Sunshine', 'WindGustDir', 'WindGustSpeed', 'WindDir9am', 'WindDir3pm', 'RainToday', 'RainTomorrow']
df_csv_cleaned = SparkFunction.remove_spaces_from_string(df_csv_cleaned,column_list)
column_list=['item_name','choice_description','item_price']
df_tsv_cleaned = SparkFunction.remove_spaces_from_string(df_tsv_cleaned,column_list)
# standardize Null values
column_dict={'Sunshine':0, 'WindSpeed9am':0, 'WindGustSpeed':0}
df_csv_cleaned = SparkFunction.standardize_null_values(df_csv_cleaned,column_dict)
column_dict={'item_price':0, 'item_name':'UNKNOWN'}
df_tsv_cleaned = SparkFunction.standardize_null_values(df_tsv_cleaned,column_dict)
# standardize datatypes
column_dict = {'Sunshine':DoubleType(), 'WindGustSpeed':DoubleType(), 'WindSpeed9am':DoubleType()}
df_csv_cleaned = SparkFunction.standardize_datatypes(df_csv_cleaned, column_dict)
column_dict = {'item_price':DoubleType()}
df_tsv_cleaned = SparkFunction.standardize_datatypes(df_tsv_cleaned, column_dict)
# clean row level duplicates
df_csv_cleaned = df_csv_cleaned.dropDuplicates()
df_tsv_cleaned = df_tsv_cleaned.dropDuplicates()
# clean unwanted records
# clean unwanted columns
####### EDA #######
# find outliers
# handle outliers


In [428]:
assignment_df = MiniAssignmentQuestions(df_csv_cleaned)
print(f"-"*45)
print(f"Mini Assignment - 1")
print(f"-"*45)
print(f"Number of days when it rained the next days: {assignment_df.days_Rained_Next_Day()} days")
print(f"Average sunshine with no rainfall: {assignment_df.avg_Sunshine_No_Rainfall()}")
print(f"Maximum temperature at 3 PM: {assignment_df.max_temp_evening()}")
print(f"Average Humidity at 3 PM when next day rain: {assignment_df.avg_evening_humidity_next_day_rain()}")
print(f"Frequent wind direction on cloudy 9 AM: {assignment_df.cloudy_day_wind_direction()}")

---------------------------------------------
Mini Assignment - 1
---------------------------------------------
Number of days when it rained the next days: 66 days
Average sunshine with no rainfall: 8.47
Maximum temperature at 3 PM: 34.5
Average Humidity at 3 PM when next day rain: 57.68
Frequent wind direction on cloudy 9 AM: SSE


In [430]:

print(f"-"*45)
print(f"Mini Assignment - 2")
print(f"-"*45)
df_tsv_cleaned = df_tsv_cleaned.withColumn('total_revenue', col('quantity') * col('item_price'))
assignment2_df = MiniAssignmentQuestions(df_tsv_cleaned)
print(f"Total Revenue: {assignment2_df.total_revenue()}")

---------------------------------------------
Mini Assignment - 2
---------------------------------------------
Total Revenue: 38914.11000000016


In [447]:
from pyspark.sql.functions import desc
df_tsv_cleaned.groupBy('item_name').agg(spark_round(spark_sum("quantity"),2).alias("count")).orderBy(desc('count')).show()

+--------------------+-----+
|           item_name|count|
+--------------------+-----+
|        Chicken Bowl|  752|
|     Chicken Burrito|  584|
| Chips and Guacamole|  501|
|       Steak Burrito|  383|
|   Canned Soft Drink|  340|
|               Chips|  227|
|          Steak Bowl|  220|
|       Bottled Water|  204|
|Chips and Fresh T...|  130|
|         Canned Soda|  124|
|  Chicken Salad Bowl|  123|
|  Chicken Soft Tacos|  116|
|       Side of Chips|  110|
|      Veggie Burrito|   97|
|    Barbacoa Burrito|   90|
|         Veggie Bowl|   87|
|       Carnitas Bowl|   71|
|       Barbacoa Bowl|   65|
|    Carnitas Burrito|   60|
|    Steak Soft Tacos|   56|
+--------------------+-----+
only showing top 20 rows



In [450]:
from pyspark.sql.functions import desc
df_tsv_cleaned.groupBy('item_name').count().select('item_name').count()

50

In [ ]:
df_tsv_cleaned.groupBy('item_name').count().select('item_name').count()

In [454]:
from pyspark.sql.functions import desc
df_tsv_cleaned.groupBy('item_name').agg(spark_round(spark_sum("total_revenue"),2).alias("total")).orderBy(desc('total')).show()

+--------------------+-------+
|           item_name|  total|
+--------------------+-------+
|        Chicken Bowl|7961.65|
|     Chicken Burrito|6320.81|
|       Steak Burrito|4203.64|
|          Steak Bowl|2470.56|
| Chips and Guacamole|2453.37|
|  Chicken Salad Bowl|1506.25|
|  Chicken Soft Tacos|1161.77|
|Chips and Fresh T...|1033.96|
|      Veggie Burrito|1002.27|
|         Veggie Bowl| 901.95|
|    Barbacoa Burrito|  885.5|
|       Carnitas Bowl| 830.71|
|       Barbacoa Bowl| 663.11|
|       Bottled Water| 638.68|
|    Carnitas Burrito| 616.33|
|   Canned Soft Drink|  590.0|
|               Chips| 573.89|
|    Steak Soft Tacos| 554.55|
|Chicken Crispy Tacos| 524.11|
|    Steak Salad Bowl| 391.15|
+--------------------+-------+
only showing top 20 rows



In [463]:
from pyspark.sql.functions import desc
df_tsv_cleaned.groupBy('order_id').agg({"quantity": "sum", "quantity": "mean"}).show()

+--------+-------------+
|order_id|avg(quantity)|
+--------+-------------+
|    1580|          1.0|
|    1829|          1.0|
|     471|          1.0|
|     496|          1.0|
|     833|          1.0|
|    1088|          1.0|
|    1342|          1.0|
|     148|          1.0|
|     463|          1.0|
|    1238|          1.0|
|    1645|          1.0|
|    1591|          2.0|
|     737|         1.25|
|    1460|          1.0|
|    1084|          1.0|
|    1483|          1.0|
|     540|          1.0|
|    1507|          1.0|
|     623|          1.0|
|     392|          1.0|
+--------+-------------+
only showing top 20 rows



In [470]:
df_tsv_cleaned.filter(col('item_name').isin('Canned Soda','6 Pack Soft Drink','Canned Soft Drink','Bottled Water')).groupBy('item_name').agg(spark_sum('total_revenue')).show()


+-----------------+------------------+
|        item_name|sum(total_revenue)|
+-----------------+------------------+
|6 Pack Soft Drink|369.93000000000023|
|Canned Soft Drink|             590.0|
|      Canned Soda|189.66000000000028|
|    Bottled Water| 638.6800000000002|
+-----------------+------------------+



In [503]:
# Read data files
input_gencsv_file='Data/Enterprise_GenAI_Adoption_Impact.csv'
#schema = SparkFunction.define_schema()
#tsv_schema = SparkFunction.define_tsv_schema(
df_csv = SparkFunction.read_csv_file(input_gencsv_file)

In [477]:
shape = (df_csv.count(), len(df_csv.columns))
print(shape)

(100000, 10)


In [480]:
df_csv.groupBy('GenAI Tool').count().select('GenAI Tool').show()

+----------+
|GenAI Tool|
+----------+
|   ChatGPT|
|    Gemini|
|     LLaMA|
|    Claude|
|      Groq|
|   Mixtral|
+----------+



In [482]:
df_csv.groupBy('Industry').count().select('Industry').show()

+--------------+
|      Industry|
+--------------+
|       Telecom|
|     Education|
| Entertainment|
|    Healthcare|
|       Finance|
|   Advertising|
|     Utilities|
|   Hospitality|
|    Technology|
|       Defense|
|Legal Services|
|        Retail|
| Manufacturing|
|Transportation|
+--------------+



In [504]:
df_csv_std = df_csv.withColumnRenamed('Company Name','company_name')\
                    .withColumnRenamed('Industry','industry')\
                    .withColumnRenamed('Country','country')\
                    .withColumnRenamed('GenAI Tool','genai_tool')\
                    .withColumnRenamed('Adoption Year','adoption_year')\
                    .withColumnRenamed('Number of Employees Impacted','number_of_employees_impacted')\
                    .withColumnRenamed('New Roles Created','new_roles_created')\
                    .withColumnRenamed('Training Hours Provided','training_hours_provided')\
                    .withColumnRenamed('Productivity Change (%)','productivity_change_percent')\
                    .withColumnRenamed('Employee Sentiment','employee_sentiment')

In [505]:
df_csv_std = df_csv_std.withColumn('adoption_year', col('adoption_year').cast(IntegerType()))\
                       .withColumn('productivity_change_percent', col('productivity_change_percent').cast(FloatType()))\
                       .withColumn('training_hours_provided', col('training_hours_provided').cast(IntegerType()))\
                       .withColumn('number_of_employees_impacted', col('number_of_employees_impacted').cast(IntegerType()))

In [512]:
df_csv_std = df_csv_std.withColumn('adoption_level', 
                                   when(col('number_of_employees_impacted')>5000,'High')
                                   .when(((col('number_of_employees_impacted')<=5000) & (col('number_of_employees_impacted')>=1000)),'Medium')
                                   .when((col('number_of_employees_impacted')<1000),'Low')
                                   .otherwise(None))

In [517]:
df_csv_std.groupBy('country','industry').agg(count('company_name').alias('total_number_of_companies')\
                                        ,avg('productivity_change_percent').alias('average_productivity_change')\
                                        ,spark_sum('new_roles_created').alias('total_new_roles_created')).show()

+-----------+--------------+-------------------------+---------------------------+-----------------------+
|    country|      industry|total_number_of_companies|average_productivity_change|total_new_roles_created|
+-----------+--------------+-------------------------+---------------------------+-----------------------+
|         UK|     Education|                      504|         17.515079372458988|                   7709|
|        USA|     Education|                      510|          18.64901960363575|                   8186|
|     Brazil|     Education|                      537|         18.544320270335874|                   8216|
|     Brazil|    Healthcare|                      553|          18.20415912010786|                   8687|
|        USA|   Hospitality|                      488|         18.550409882283603|                   7483|
|        USA|       Finance|                      506|         18.288142327263422|                   7989|
|     France|Transportation|         

In [520]:
df_csv_std.groupBy('adoption_year').agg(count('company_name').alias('number_of_companies')\
                                       ,avg('training_hours_provided').alias('average_training_hours')\
                                       ,mode('genai_tool').alias('most_used_gen_ai_tool')).show()

+-------------+-------------------+----------------------+---------------------+
|adoption_year|number_of_companies|average_training_hours|most_used_gen_ai_tool|
+-------------+-------------------+----------------------+---------------------+
|         2023|              33344|    12717.287817898272|                 Groq|
|         2022|              33180|    12797.869469559975|                 Groq|
|         2024|              33476|    12712.635709164775|               Gemini|
+-------------+-------------------+----------------------+---------------------+



In [510]:
df_csv_std.printSchema()

root
 |-- company_name: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- country: string (nullable = true)
 |-- genai_tool: string (nullable = true)
 |-- adoption_year: integer (nullable = true)
 |-- number_of_employees_impacted: integer (nullable = true)
 |-- new_roles_created: integer (nullable = true)
 |-- training_hours_provided: integer (nullable = true)
 |-- productivity_change_percent: float (nullable = true)
 |-- employee_sentiment: string (nullable = true)
 |-- adoption_level: string (nullable = true)



In [513]:
df_csv_std.show()

+--------------------+--------------+------------+----------+-------------+----------------------------+-----------------+-----------------------+---------------------------+--------------------+--------------+
|        company_name|      industry|     country|genai_tool|adoption_year|number_of_employees_impacted|new_roles_created|training_hours_provided|productivity_change_percent|  employee_sentiment|adoption_level|
+--------------------+--------------+------------+----------+-------------+----------------------------+-----------------+-----------------------+---------------------------+--------------------+--------------+
| Davis LLC Pvt. Ltd.|    Healthcare|         USA|   Mixtral|         2022|                        5277|                8|                    657|                       25.2|Productivity incr...|          High|
|Roberts, Holland ...|       Telecom|South Africa|    Claude|         2023|                       18762|               17|                  23021|          